In [71]:
import os
import pandas as pd

In [72]:
# Pasta com os arquivos
BASE_DIR = r'C:\Users\lucas\Desktop\TCC\TCC-Lucas-Garcia'
RAW_DIR = os.path.join(BASE_DIR, 'data', 'ccee', 'geracao_usina')

# Lista para armazenar tipos únicos
tipos_unicos = set()

# Verificar se o diretório existe antes de iterar
if os.path.exists(RAW_DIR):
    # Iterar sobre os arquivos na pasta
    for arquivo in os.listdir(RAW_DIR):
        if arquivo.endswith('.csv'):
            caminho = os.path.join(RAW_DIR, arquivo)
            df = pd.read_csv(caminho, sep=';', encoding='utf-8', on_bad_lines='skip')
            if 'FONTE_ENERGIA_PRIMARIA' in df.columns:
                tipos_unicos.update(df['FONTE_ENERGIA_PRIMARIA'].unique())
else:
    print(f"Diretório não encontrado: {RAW_DIR}")

# Exibir tipos únicos
print("Tipos únicos de usina:", list(tipos_unicos))

Tipos únicos de usina: ['Hidráulica', 'Térmica - Outros', 'Térmica a GNL', 'Térmica a Gás', 'Eólica', 'Térmica Solar', 'Térmica a Biomassa', 'Hidráulica CGH', 'Hidráulica PCH', 'Solar Fotovoltaica', 'Térmica Reação Exotérmica', 'Térmica a Óleo', 'Térmica bi-Combustível - gás/óleo', 'Térmica Nuclear', 'Térmica a Carvão Mineral']


Quero os tipos: Hidráulica PCH (SE ou S), Eólica (NE), Térmica a Biomassa

In [73]:
# Dados completos
df_2024 = pd.read_csv(os.path.join(RAW_DIR, 'parcela_usina_montante_mensal_2024.csv'), sep=';', encoding='utf-8', on_bad_lines='skip')
df_2025 = pd.read_csv(os.path.join(RAW_DIR, 'parcela_usina_montante_mensal_2025.csv'), sep=';', encoding='utf-8', on_bad_lines='skip')

mapa_sub = {'SUDESTE': 'SE', 'SUL': 'S', 'NORDESTE': 'NE', 'NORTE': 'N'}
df_2024['SUBMERCADO'] = df_2024['SUBMERCADO'].map(mapa_sub).fillna(df_2024['SUBMERCADO'])
df_2025['SUBMERCADO'] = df_2025['SUBMERCADO'].map(mapa_sub).fillna(df_2025['SUBMERCADO'])

filtros = [
    {'nome': 'Hidráulica PCH (SE/S)', 'tipo': 'Hidráulica PCH', 'submercados': ['SE', 'S'], 'cap_t': (3, 30), 'gf': (1.5, 18)},
    {'nome': 'Eólica (NE)', 'tipo': 'Eólica', 'submercados': ['NE'], 'cap_t': (30, 120), 'gf': (15, 50)},
    {'nome': 'Térmica a Biomassa', 'tipo': 'Térmica a Biomassa', 'submercados': None, 'cap_t': (30, 80), 'gf': (10, 40)}
]

for filtro in filtros:
    mask_2024 = df_2024['FONTE_ENERGIA_PRIMARIA'] == filtro['tipo']
    mask_2025 = df_2025['FONTE_ENERGIA_PRIMARIA'] == filtro['tipo']
    if filtro['submercados']:
        mask_2024 &= df_2024['SUBMERCADO'].isin(filtro['submercados'])
        mask_2025 &= df_2025['SUBMERCADO'].isin(filtro['submercados'])
    
    count_2024 = df_2024[mask_2024].groupby('COD_ATIVO')['MES_REFERENCIA'].nunique()
    count_2025 = df_2025[mask_2025].groupby('COD_ATIVO')['MES_REFERENCIA'].nunique()
    count_total = count_2024.add(count_2025, fill_value=0).astype(int)
    max_meses = count_total.max() if len(count_total) > 0 else 0
    completas = count_total[count_total == max_meses].index
    
    df_filtrado = pd.concat([df_2024[mask_2024], df_2025[mask_2025]])
    df_filtrado = df_filtrado[df_filtrado['COD_ATIVO'].isin(completas)]
    
    cap_min, cap_max = filtro['cap_t']
    gf_min, gf_max = filtro['gf']
    df_range = df_filtrado[(df_filtrado['CAP_T'] >= cap_min) & (df_filtrado['CAP_T'] <= cap_max) & 
                           (df_filtrado['GARANTIA_FISICA'] >= gf_min) & (df_filtrado['GARANTIA_FISICA'] <= gf_max)]
    
    print(f"\n{filtro['nome']}:")
    if len(df_range) > 0:
        print(f"  CAP_T: min={df_range['CAP_T'].min():.1f} MW, max={df_range['CAP_T'].max():.1f} MW")
        print(f"  GARANTIA_FISICA: min={df_range['GARANTIA_FISICA'].min():.1f} MWm, max={df_range['GARANTIA_FISICA'].max():.1f} MWm")
        print(f"  Usinas no range: {df_range['COD_ATIVO'].nunique()}")
    else:
        print(f"  Nenhuma usina no range especificado")



Hidráulica PCH (SE/S):
  CAP_T: min=3.0 MW, max=30.0 MW
  GARANTIA_FISICA: min=1.5 MWm, max=17.9 MWm
  Usinas no range: 281

Eólica (NE):
  CAP_T: min=30.0 MW, max=83.2 MW
  GARANTIA_FISICA: min=15.0 MWm, max=44.7 MWm
  Usinas no range: 344

Térmica a Biomassa:
  CAP_T: min=30.0 MW, max=80.0 MW
  GARANTIA_FISICA: min=10.0 MWm, max=30.9 MWm
  Usinas no range: 90


In [74]:
# Buscar usinas com GARANTIA_FISICA = 16.7 por tipo
for filtro in filtros:
    mask_2024 = df_2024['FONTE_ENERGIA_PRIMARIA'] == filtro['tipo']
    mask_2025 = df_2025['FONTE_ENERGIA_PRIMARIA'] == filtro['tipo']
    if filtro['submercados']:
        mask_2024 &= df_2024['SUBMERCADO'].isin(filtro['submercados'])
        mask_2025 &= df_2025['SUBMERCADO'].isin(filtro['submercados'])
    
    count_2024 = df_2024[mask_2024].groupby('COD_ATIVO')['MES_REFERENCIA'].nunique()
    count_2025 = df_2025[mask_2025].groupby('COD_ATIVO')['MES_REFERENCIA'].nunique()
    count_total = count_2024.add(count_2025, fill_value=0).astype(int)
    max_meses = count_total.max() if len(count_total) > 0 else 0
    completas = count_total[count_total == max_meses].index
    
    df_filtrado = pd.concat([df_2024[mask_2024], df_2025[mask_2025]])
    df_filtrado = df_filtrado[df_filtrado['COD_ATIVO'].isin(completas)]
    df_167 = df_filtrado[df_filtrado['GARANTIA_FISICA'].round(1) == 16.7]
    usinas_167 = df_167.groupby('COD_ATIVO')['SIGLA_ATIVO'].first().reset_index()
    
    print(f"\n{filtro['nome']} - GF=16.7 MWm: {len(usinas_167)}")
    for _, row in usinas_167.iterrows():
        print(f"  {row['COD_ATIVO']} - {row['SIGLA_ATIVO']}")



Hidráulica PCH (SE/S) - GF=16.7 MWm: 2
  8605 - PORTOBELLO-C. ENCANO
  110222 - PCH PARANATINGA II

Eólica (NE) - GF=16.7 MWm: 7
  81285 - SANTO ONOFRE III
  82613 - DAMASCENA
  97523 - VENTOS S VIRGILIO 02
  107215 - V STA ANGELA 03
  126003 - SANTA ANGELA 18
  145070 - AURA QUEIMADA NOVA 1
  151894 - EOL Chafariz 5

Térmica a Biomassa - GF=16.7 MWm: 1
  8578 - PIONEIROS II
